In [ ]:
# Class map: Maps annotation labels (e.g., 'PC') to YOLO class IDs (0-10)
class_map = {
    "BL": 0,  # Bus Lane
    "CL": 1,  # Cycle Lane
    "DM": 2,  # Diamond
    "JB": 3,  # Junction Box
    "LA": 4,  # Left Arrow
    "PC": 5,  # Pedestrian Crossing
    "RA": 6,  # Right Arrow
    "SA": 7,  # Straight Arrow
    "SL": 8,  # Slow
    "SLA": 9,  # Straight-Left Arrow
    "SRA": 10,  # Straight-Right Arrow
}

# Optional: Color map for visualization (RGB tuples as provided)
color_map = {
    "BL": (0, 255, 255),
    "CL": (0, 128, 255),
    "DM": (178, 102, 255),
    "JB": (255, 255, 51),
    "LA": (255, 102, 178),
    "PC": (255, 255, 0),
    "RA": (255, 0, 127),
    "SA": (255, 0, 255),
    "SL": (0, 255, 0),
    "SLA": (255, 128, 0),
    "SRA": (255, 0, 0),
}

# Print to verify
print("Class Map:", class_map)
print("Color Map:", color_map)

converting polygon to yolo

In [ ]:
import os
import json
from PIL import Image

def convert_polygons_to_yolo(json_dir, img_dir, label_dir, class_map):
    os.makedirs(label_dir, exist_ok=True)
    for json_file in os.listdir(json_dir):
        if not json_file.endswith('.json'): continue
        json_path = os.path.join(json_dir, json_file)
        with open(json_path, 'r') as f:
            data = json.load(f)
        
        img_id = os.path.splitext(json_file)[0]
        img_path = os.path.join(img_dir, f'{img_id}.jpg')  # Adjust if extensions vary
        if not os.path.exists(img_path): continue
        
        img = Image.open(img_path)
        width, height = img.size
        
        txt_path = os.path.join(label_dir, f'{img_id}.txt')
        with open(txt_path, 'w') as txt_f:
            for shape in data['shapes']:
                class_label = shape['label']  # e.g., 'PC'
                class_id = class_map.get(class_label, -1)  # -1 if unknown (error handling)
                if class_id == -1:
                    print(f"Warning: Unknown class '{class_label}' in {json_file}")
                    continue
                points = shape['points']
                # Normalize and flatten: x1 y1 x2 y2 ...
                normalized = [coord / width if i % 2 == 0 else coord / height for point in points for i, coord in enumerate(point)]
                txt_f.write(f"{class_id} {' '.join(map(str, normalized))}\n")

# Convert train (pass the class_map)
base_path = './'
convert_polygons_to_yolo(
    os.path.join(base_path, 'train/polygon_annotations'),
    os.path.join(base_path, 'train/images'),
    os.path.join(base_path, 'train/labels'),  # Creates/overwrites this folder
    class_map
)

# Repeat for test
convert_polygons_to_yolo(
    os.path.join(base_path, 'test/polygon_annotations'),
    os.path.join(base_path, 'test/images'),
    os.path.join(base_path, 'test/labels'),  # Creates/overwrites this folder
    class_map
)

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt

model = YOLO("yolov8n-seg.pt")  # Your trained model

results = model.train(
    data=os.path.join(base_path, "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    name="road_lines",
    device="cpu",
)